# Chess Move Transformer — v3
### Board-state + metadata conditioned decoder-only transformer · 2200–2600 ELO Lichess

---

## What changed from v2, and why

v2 reached **~29% masked Top-1** and overfit (val loss reversing while train kept falling). Two
separate problems, addressed separately here.

| Change | Rationale |
|---|---|
| **Board planes fed at every position** | v1's `unmasked_argmax_legal_rate` was **0.539** — the model's own top pick was illegal 46% of the time, i.e. a large share of capacity was going into *tracking* the board instead of *choosing* a move. The board is a deterministic function of the move history, so this adds no information — it is an inductive bias that hands the model the tracking for free. |
| **Full position state, not just piece placement** | Side-to-move, the 4 castling rights, and the en-passant file. Piece placement alone is an *ambiguous* position. |
| **Per-position, per-side ELO** | v2 conditioned on `(white+black)/2`, one scalar per game — that blurs exactly the signal we want. v3 feeds the **side-to-move's** rating and the opponent's, and it alternates every ply. |
| **Derived board features** | Attack maps, legal from/to squares, last-move squares, check & halfmove clock. The model projects a *flat* vector — it has no convolution, no spatial prior — so "which squares does Black attack" is genuinely expensive for it to recompute from raw planes every layer. See Section 5. |
| **Time-control conditioning** | `TimeControl` (base + increment) is in every game header. A 180+0 blitz game and a 600+5 rapid game have visibly different human move distributions. |
| **Clock features (if present in the PGN)** | Time pressure is strongly predictive and not derivable from the board. The Lichess **Elite** database strips `%clk` comments, so this auto-disables to zeros with a `has_clock` flag. Raw Lichess monthly dumps *do* carry them, so the code path stays — it costs 3 always-zero input dims and means no code change if you later switch data sources. |
| **Multi-file PGN input + lightweight records** | `PGN_PATHS` is a list; drop in more months. Games are parsed into small dicts instead of held as `chess.pgn.Game` objects, so RAM stops being the wall on data scaling. |
| **Batched teacher-forced evaluation** | v1/v2 did one forward pass *per position*. Evaluation is teacher-forced on the real move history, and causal masking means position `t`'s logits already are the prediction for `t+1` — so **one forward pass per game** yields every position at once. ~40x faster, which is what makes evaluating on enough positions to resolve small differences practical. |
| **Higher dropout + early stopping** | Damage control for the overfit, not a fix. The fix is `MAX_GAMES`. |

## The one thing this notebook does not fix

**Overfitting at 12k games is a data problem.** 12,000 games x ~82 plies is ~1M training tokens for
a ~6M parameter model — about **0.2 tokens per parameter**, when the rough compute-optimal target is
~20. Board planes and metadata will help, but they do not manufacture data. Raise `MAX_GAMES` and add
PGN files to `PGN_PATHS` as soon as you have them; that is still the single biggest lever available.

## Deliberately *not* changed

Color-relative board flipping and a factorized `from_square x to_square` head are both likely wins,
but they are held back so this run is a clean ablation of **board state + metadata** against v2.
Bundling four changes at once tells you the total but not which part earned it.

---

## Notebook Structure

| # | Section |
|---|---|
| 1 | Setup & Config |
| 2 | Load Games -> Lightweight Records (multi-PGN, clock extraction) |
| 3 | Game-Level Split |
| 4 | Move Vocabulary (unchanged — full UCI action space) |
| 5 | Feature Construction (board planes + metadata) |
| 6 | Dataset & DataLoader |
| 7 | Model — Board/Meta-Conditioned Decoder-Only Transformer |
| 8 | Training |
| 9 | Batched Legal-Move-Masked Evaluation |
| 10 | Save Artifacts |
| 11 | Summary & Next Steps |

---
# 1. Setup & Config

In [ ]:
import os
import re
import json
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import chess
import chess.pgn
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print("Using device:", device)

In [ ]:
# ----- Data -------------------------------------------------------------
# Add more months here as you download them -- this is the highest-leverage knob
# in the notebook. Overfitting at 12k training games is a data problem.
PGN_PATHS  = ["lichess_elite_2019-10.pgn"]

MIN_ELO, MAX_ELO   = 2200, 2600
MAX_GAMES          = 15000     # <-- raise this
MIN_MOVES, MAX_MOVES = 8, 200
BLOCK_SIZE         = 160       # max context in plies

# ----- Features ---------------------------------------------------------
# Toggles so each block can be ablated without touching the model code.
# Costs below are measured CPU time per position, incurred in the DataLoader
# (overlapped with the GPU when NUM_WORKERS > 0).
USE_ATTACK_MAPS   = True   # +128 dims, ~13 us/pos
USE_LEGAL_FROM_TO = True   # +128 dims, ~40 us/pos  <- most expensive, likely most useful
USE_LAST_MOVE     = True   # +128 dims, free (read off the move stack)
USE_TACTICAL_FLAGS = True  # +2 dims,   ~1 us/pos

# ----- Model ------------------------------------------------------------
D_MODEL, N_HEAD, N_LAYER = 256, 4, 4
EMB_INIT_STD             = 0.02  # GPT-2 style; also the starting gain on the side-channels

# Kept at v2's value ON PURPOSE. v3 vs v2 should differ by the board/metadata
# features and nothing else -- changing regularization at the same time makes a
# drop impossible to attribute. Raise it only as a separate, isolated experiment.
DROPOUT                  = 0.1

# ----- Training ---------------------------------------------------------
BATCH_SIZE           = 64
LR                   = 3e-4
WEIGHT_DECAY         = 0.01
N_EPOCHS             = 30
EARLY_STOP_PATIENCE  = 4         # epochs without val improvement before stopping

# DataLoader workers. Board planes are built on the fly by replaying each game,
# so workers overlap that CPU work with the GPU. Keep 0 on Windows/Jupyter
# (spawned workers re-import __main__ and break); 4 is a good value on Mac/Linux.
NUM_WORKERS = 0

# ----- Evaluation -------------------------------------------------------
EVAL_MAX_GAMES = 1500   # ~120k positions; None = all. Batched eval makes this cheap.

# ----- Output -----------------------------------------------------------
# Deliberately NOT the v1/v2 filenames -- those would be overwritten in place.
CKPT_PATH  = "chess_gpt_v3.pt"
VOCAB_PATH = "chess_move_vocab_v3.json"

---
# 2. Load Games → Lightweight Records

Two changes from v1/v2 beyond reading multiple files.

**Records, not `Game` objects.** v1/v2 kept 15,000 parsed `chess.pgn.Game` objects alive for the
whole session. Those are heavy, and they are what caps how much data this pipeline can hold. Each
game is reduced here to a small dict — UCI move strings, two ELOs, a clock list — which is what
every later stage actually needs.

**Clock extraction.** Lichess PGNs annotate each move with `[%clk H:MM:SS]`, the mover's time
*remaining after making that move*. Some derived databases strip these comments; the cell reports
what fraction of games actually had them, and if the answer is 0% the clock features below
auto-disable via a flag rather than silently feeding zeros as if they were real readings.

In [ ]:
CLK_RE = re.compile(r"\[%clk\s+(\d+):(\d+):(\d+(?:\.\d+)?)\]")
TC_RE  = re.compile(r"^(\d+)(?:\+(\d+))?")


def parse_clock(comment):
    """Seconds remaining, from a [%clk H:MM:SS] comment. None if absent."""
    m = CLK_RE.search(comment or "")
    if not m:
        return None
    return int(m.group(1)) * 3600 + int(m.group(2)) * 60 + float(m.group(3))


def parse_time_control(headers):
    """(base_seconds, increment) from a "180+0"-style header. (None, 0.0) if unparseable."""
    m = TC_RE.match(headers.get("TimeControl", "") or "")
    if not m:
        return None, 0.0
    base = float(m.group(1))
    inc = float(m.group(2)) if m.group(2) else 0.0
    return (base if base > 0 else None), inc


def game_to_record(game):
    """chess.pgn.Game -> compact dict, or None if it fails the filters."""
    try:
        w_elo = int(game.headers.get("WhiteElo", 0))
        b_elo = int(game.headers.get("BlackElo", 0))
    except ValueError:
        return None

    if not (MIN_ELO <= w_elo <= MAX_ELO and MIN_ELO <= b_elo <= MAX_ELO):
        return None

    ucis, clocks = [], []
    for node in game.mainline():
        ucis.append(node.move.uci())
        clocks.append(parse_clock(node.comment))

    if not (MIN_MOVES <= len(ucis) <= MAX_MOVES):
        return None

    base, inc = parse_time_control(game.headers)

    # NOTE: Result / WhiteRatingDiff / BlackRatingDiff / Termination / ECO / Opening
    # are deliberately NOT read. See the leakage note in Section 5.
    return {
        "uci":  ucis,
        "w":    w_elo,
        "b":    b_elo,
        "clk":  clocks,
        "base": base,
        "inc":  inc,
    }


records = []
t0 = time.time()

for path in PGN_PATHS:
    if not os.path.exists(path):
        print(f"!! Missing PGN, skipping: {path}")
        continue

    print(f"Reading {path} ...")
    with open(path, encoding="utf-8", errors="replace") as f:
        pbar = tqdm(total=MAX_GAMES, initial=len(records), desc=os.path.basename(path))
        while len(records) < MAX_GAMES:
            game = chess.pgn.read_game(f)
            if game is None:
                break
            rec = game_to_record(game)
            if rec is not None:
                records.append(rec)
                pbar.update(1)
        pbar.close()

    if len(records) >= MAX_GAMES:
        break

n_with_clocks = sum(1 for r in records if r["base"] and any(c is not None for c in r["clk"]))
HAS_CLOCK_DATA = n_with_clocks > 0

print(f"\nGames loaded: {len(records)}  ({time.time() - t0:.1f}s)")
print(f"ELO range: {MIN_ELO}-{MAX_ELO}")
print(f"Games with usable clock data: {n_with_clocks} ({100 * n_with_clocks / max(1, len(records)):.1f}%)")
if not HAS_CLOCK_DATA:
    print("  -> No %clk comments (expected for the Lichess Elite DB, which strips them).")
    print("     Clock columns feed 0 with has_clock=0, so the model can tell 'unknown'")
    print("     apart from 'no time left'. Raw Lichess monthly dumps do carry %clk.")

from collections import Counter
tc_counts = Counter((r["base"], r["inc"]) for r in records)
print("\nTime controls present (base+inc -> games):")
for (base, inc), n in tc_counts.most_common(6):
    label = f"{int(base)}+{int(inc)}" if base else "unknown"
    print(f"  {label:>12}  {n}")
if len(tc_counts) == 1:
    print("  -> Single time control: the TC features are constant and will contribute nothing.")
    print("     They start paying off once you mix bullet/blitz/rapid into PGN_PATHS.")

---
# 3. Game-Level Train/Test Split

Split by **game**, not by position, exactly as in v1/v2 — otherwise positions from the same game
leak across the split and every accuracy number below becomes meaningless. Same `SEED`, so pointing
this at the same PGN reproduces the same split as the earlier notebooks.

In [ ]:
train_records, test_records = train_test_split(records, test_size=0.2, random_state=SEED)

print(f"Train games: {len(train_records)}")
print(f"Test games:  {len(test_records)}")

---
# 4. Move Vocabulary — Full UCI Action Space

Unchanged from v1/v2, and deliberately so: the vocabulary enumerates every UCI string
`python-chess` can emit over all square pairs (plus the four promotion suffixes on the promotion
ranks), so **every legal move in every legal position has a slot** whether or not it appeared in
training. That is what makes legal-move masking always well-defined. Do not replace this with a
frequency-derived vocabulary.

In [ ]:
PROMOTION_PIECES = [chess.QUEEN, chess.ROOK, chess.BISHOP, chess.KNIGHT]


def generate_full_uci_vocab():
    moves = set()
    for from_sq in chess.SQUARES:
        for to_sq in chess.SQUARES:
            if from_sq == to_sq:
                continue
            from_rank, to_rank = chess.square_rank(from_sq), chess.square_rank(to_sq)
            is_promotion_rank = (from_rank == 6 and to_rank == 7) or (from_rank == 1 and to_rank == 0)

            moves.add(chess.Move(from_sq, to_sq).uci())
            if is_promotion_rank:
                for promo in PROMOTION_PIECES:
                    moves.add(chess.Move(from_sq, to_sq, promotion=promo).uci())
    return sorted(moves)


SPECIAL_TOKENS = ["<PAD>", "<BOS>"]

itos = SPECIAL_TOKENS + generate_full_uci_vocab()
stoi = {tok: i for i, tok in enumerate(itos)}

PAD_ID = stoi["<PAD>"]
BOS_ID = stoi["<BOS>"]
VOCAB_SIZE = len(itos)

print(f"Vocabulary size: {VOCAB_SIZE}")

# Encode each record's move list once: [<BOS>, m1, m2, ...]
for r in records:
    r["ids"] = [BOS_ID] + [stoi[u] for u in r["uci"]]

lengths = [len(r["ids"]) for r in train_records]
print(f"Sequence length (plies) -- min: {min(lengths)}, max: {max(lengths)}, mean: {np.mean(lengths):.1f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=40, edgecolor="black", alpha=0.7)
plt.axvline(np.mean(lengths), color="red", linestyle="--", label=f"Mean: {np.mean(lengths):.0f}")
plt.title("Game length in half-moves (plies) - Train set")
plt.xlabel("Plies"); plt.ylabel("Frequency"); plt.legend()
plt.show()

---
# 5. Feature Construction — Board Planes + Metadata

## Alignment (the easy thing to get wrong)

With `ids = [BOS, m1, m2, ..., mn]`, the model input is `x = ids[:-1]` and the target is
`y = ids[1:]`. So input position `t` must be paired with the board **as it stands before move
`t+1` is played**:

| position `t` | input token | predicts | board features must be |
|---|---|---|---|
| 0 | `<BOS>` | `m1` | the **starting** position |
| 1 | `m1` | `m2` | position after `m1` |
| `t` | `m_t` | `m_{t+1}` | position after `m1..m_t` |

The loop below therefore records the features **before** each `board.push()`. Off-by-one here leaks
the answer into the input and produces a beautiful, meaningless accuracy number.

## Board vector (up to 1167 dims)

| Slice | Dims | Contents | Toggle |
|---|---|---|---|
| Piece planes | 768 | 12 planes (6 piece types x 2 colors) x 64 squares | always |
| Side to move | 1 | 1.0 if White to move | always |
| Castling rights | 4 | W kingside, W queenside, B kingside, B queenside | always |
| En-passant file | 8 | one-hot, all zero if no en-passant square | always |
| Attack maps | 128 | squares attacked by White (64), by Black (64) | `USE_ATTACK_MAPS` |
| Legal from/to | 128 | squares a legal move can start from (64) and land on (64) | `USE_LEGAL_FROM_TO` |
| Last move | 128 | previous move's from-square (64) and to-square (64) | `USE_LAST_MOVE` |
| Tactical flags | 2 | side-to-move in check; halfmove clock / 100 | `USE_TACTICAL_FLAGS` |

**Why derived features at all, when they are functions of the piece planes?** Because of how this
model consumes them. Board input enters through a single `nn.Linear` over a *flat* 1167-vector —
there is no convolution and no spatial prior anywhere in the architecture. Recomputing "which
squares does Black attack" from raw planes requires sliding-piece ray logic that a 4-layer
transformer over flat inputs is poorly shaped to express. Handing it over costs microseconds of CPU
and frees capacity. Same argument as the board planes themselves: no new information, better
inductive bias.

`USE_LEGAL_FROM_TO` is the expensive one (~40 us/position) and probably the most valuable — it
tells the model which pieces can actually move at all, so none of its capacity goes into
rediscovering legality. This is legitimate, not leakage: it is a function of the *current* position,
exactly what the human at the board can see.

Planes are unpacked from `python-chess`'s internal bitboard integers rather than looping over 64
squares, which keeps on-the-fly construction cheap enough to do inside the DataLoader.

## Metadata vector (7 dims)

`[elo_stm, elo_opponent, clock_stm, clock_opponent, has_clock, tc_base, tc_increment]`, normalized.
ELO alternates every ply, which is the point — v2's single per-game average could not express "a
2600 is on move here."

**Clock causality.** A `%clk` reading is stamped on a move *after* it is made, so the value on move
`t+1` reveals how long the player thought — using it would leak. The mover's last *observable*
reading is the one on their previous move, ply `t-2`. Hence the lag below. With the Elite database
these columns are all zero and `has_clock` is 0.

## Headers deliberately NOT used

The PGN headers carry several fields that look like free features and are actually **future
information**. Feeding any of these inflates the metrics into meaninglessness:

| Header | Why it leaks |
|---|---|
| `Result` | The outcome of the game being predicted. |
| `WhiteRatingDiff` / `BlackRatingDiff` | A deterministic function of `Result`. |
| `Termination` | "Time forfeit" reveals how the game ended, and correlates with the losing side's play. |
| `ECO` / `Opening` | Determined by moves not yet played. At ply 0, knowing the game is an "Indian Game" tells the model the answer to its first several predictions. |

`TimeControl`, `WhiteElo` and `BlackElo` are the safe ones: all three are known before the first
move is played.

In [ ]:
# 12 planes: white P,N,B,R,Q,K then black P,N,B,R,Q,K
PLANE_SPEC = [
    (color, pt)
    for color in (chess.WHITE, chess.BLACK)
    for pt in (chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING)
]

N_PIECE_PLANES = len(PLANE_SPEC)          # 12

# Layout offsets, so every slice has one authoritative definition.
OFF_PLANES  = 0
OFF_STM     = OFF_PLANES + N_PIECE_PLANES * 64      # 768
OFF_CASTLE  = OFF_STM + 1                           # 769
OFF_EP      = OFF_CASTLE + 4                        # 773
_next       = OFF_EP + 8                            # 781

OFF_ATTACK = _next
if USE_ATTACK_MAPS:
    _next += 128
OFF_LEGAL = _next
if USE_LEGAL_FROM_TO:
    _next += 128
OFF_LASTMOVE = _next
if USE_LAST_MOVE:
    _next += 128
OFF_FLAGS = _next
if USE_TACTICAL_FLAGS:
    _next += 2

BOARD_DIM = _next
META_DIM  = 7

_plane_buf = np.zeros((N_PIECE_PLANES, 8), dtype=np.uint8)
_bb_buf    = np.zeros((2, 8), dtype=np.uint8)


def _bits_from_mask(mask):
    """64-bit bitboard int -> (64,) float array, bit i == square i (a1=0, h8=63)."""
    return np.unpackbits(
        np.frombuffer(np.uint64(mask).tobytes(), dtype=np.uint8), bitorder="little"
    )


def board_feature_vector(board, out=None):
    """Full-position feature vector. NOT just piece placement -- see Section 5 table."""
    if out is None:
        out = np.zeros(BOARD_DIM, dtype=np.float32)

    for i, (color, pt) in enumerate(PLANE_SPEC):
        _plane_buf[i] = np.frombuffer(
            np.uint64(board.pieces_mask(pt, color)).tobytes(), dtype=np.uint8
        )
    out[OFF_PLANES:OFF_STM] = np.unpackbits(
        _plane_buf, axis=1, bitorder="little"
    ).reshape(-1)

    out[OFF_STM] = 1.0 if board.turn == chess.WHITE else 0.0
    out[OFF_CASTLE + 0] = float(board.has_kingside_castling_rights(chess.WHITE))
    out[OFF_CASTLE + 1] = float(board.has_queenside_castling_rights(chess.WHITE))
    out[OFF_CASTLE + 2] = float(board.has_kingside_castling_rights(chess.BLACK))
    out[OFF_CASTLE + 3] = float(board.has_queenside_castling_rights(chess.BLACK))

    out[OFF_EP:OFF_EP + 8] = 0.0
    if board.ep_square is not None:
        out[OFF_EP + chess.square_file(board.ep_square)] = 1.0

    if USE_ATTACK_MAPS:
        for j, color in enumerate((chess.WHITE, chess.BLACK)):
            mask = 0
            for sq in chess.scan_forward(board.occupied_co[color]):
                mask |= board.attacks_mask(sq)
            out[OFF_ATTACK + j * 64 : OFF_ATTACK + (j + 1) * 64] = _bits_from_mask(mask)

    if USE_LEGAL_FROM_TO:
        from_mask = to_mask = 0
        for mv in board.legal_moves:
            from_mask |= 1 << mv.from_square
            to_mask |= 1 << mv.to_square
        out[OFF_LEGAL:OFF_LEGAL + 64] = _bits_from_mask(from_mask)
        out[OFF_LEGAL + 64:OFF_LEGAL + 128] = _bits_from_mask(to_mask)

    if USE_LAST_MOVE:
        out[OFF_LASTMOVE:OFF_LASTMOVE + 128] = 0.0
        if board.move_stack:
            last = board.move_stack[-1]
            out[OFF_LASTMOVE + last.from_square] = 1.0
            out[OFF_LASTMOVE + 64 + last.to_square] = 1.0

    if USE_TACTICAL_FLAGS:
        out[OFF_FLAGS] = float(board.is_check())
        out[OFF_FLAGS + 1] = min(1.0, board.halfmove_clock / 100.0)

    return out


def build_board_features(ids, T):
    """Replay the game, capturing features BEFORE each move. Returns (T, BOARD_DIM)."""
    feats = np.zeros((T, BOARD_DIM), dtype=np.float32)
    board = chess.Board()
    for t in range(T):
        board_feature_vector(board, out=feats[t])
        board.push(chess.Move.from_uci(itos[ids[t + 1]]))
    return feats


ELO_SPAN = max(1, MAX_ELO - MIN_ELO)
_TC_LOG_MAX = math.log1p(10800.0)   # 3h, so normal blitz/rapid spread over most of [0,1]


def build_meta_features(rec, T):
    """(T, META_DIM): per-ply per-side ELO, causally-lagged clocks, time control."""
    meta = np.zeros((T, META_DIM), dtype=np.float32)

    e_w = (rec["w"] - MIN_ELO) / ELO_SPAN
    e_b = (rec["b"] - MIN_ELO) / ELO_SPAN

    base, clks = rec["base"], rec["clk"]
    has_clk = 1.0 if (base and any(c is not None for c in clks)) else 0.0

    # Time control is constant across the game. Log-scaled: the difference between
    # 60s and 180s matters far more to move quality than 1800s vs 1920s.
    tc_base = min(1.0, math.log1p(base) / _TC_LOG_MAX) if base else 0.0
    tc_inc = min(1.0, rec.get("inc", 0.0) / 60.0)

    for t in range(T):
        white_to_move = (t % 2 == 0)
        meta[t, 0] = e_w if white_to_move else e_b     # side to move
        meta[t, 1] = e_b if white_to_move else e_w     # opponent

        if has_clk:
            # Mover's last observable reading is on their own previous move (t-2).
            # Opponent's is on the immediately preceding ply (t-1). Before those
            # exist, the player is still on the full base time.
            c_stm = clks[t - 2] if t >= 2 else base
            c_opp = clks[t - 1] if t >= 1 else base
            if c_stm is not None:
                meta[t, 2] = min(1.0, c_stm / base)
            if c_opp is not None:
                meta[t, 3] = min(1.0, c_opp / base)

        meta[t, 4] = has_clk
        meta[t, 5] = tc_base
        meta[t, 6] = tc_inc

    return meta


print(f"BOARD_DIM = {BOARD_DIM}, META_DIM = {META_DIM}")
print(f"  piece planes    768   [{OFF_PLANES}:{OFF_STM}]")
print(f"  stm/castle/ep    13   [{OFF_STM}:{OFF_EP + 8}]")
print(f"  attack maps     {128 if USE_ATTACK_MAPS else 0:3d}   " + (f"[{OFF_ATTACK}:{OFF_ATTACK+128}]" if USE_ATTACK_MAPS else "(off)"))
print(f"  legal from/to   {128 if USE_LEGAL_FROM_TO else 0:3d}   " + (f"[{OFF_LEGAL}:{OFF_LEGAL+128}]" if USE_LEGAL_FROM_TO else "(off)"))
print(f"  last move       {128 if USE_LAST_MOVE else 0:3d}   " + (f"[{OFF_LASTMOVE}:{OFF_LASTMOVE+128}]" if USE_LAST_MOVE else "(off)"))
print(f"  tactical flags  {2 if USE_TACTICAL_FLAGS else 0:3d}   " + (f"[{OFF_FLAGS}:{OFF_FLAGS+2}]" if USE_TACTICAL_FLAGS else "(off)"))

_t0 = time.perf_counter()
_probe_board = chess.Board()
for _ in range(200):
    board_feature_vector(_probe_board)
print(f"\nFeature build cost: {(time.perf_counter() - _t0) / 200 * 1e6:.0f} us/position")

In [ ]:
# --- Sanity check: features must describe the position BEFORE the predicted move ---
_r = train_records[0]
_ids = _r["ids"][: BLOCK_SIZE + 1]
_T = len(_ids) - 1
_bf = build_board_features(_ids, _T)

# Position 0 must be the untouched starting position.
assert np.array_equal(_bf[0], board_feature_vector(chess.Board())), "position 0 is not the start position"

# Position t must equal the board after replaying exactly t moves.
_probe = chess.Board()
for _t in range(min(12, _T)):
    assert np.array_equal(_bf[_t], board_feature_vector(_probe)), f"misaligned board features at t={_t}"
    _probe.push(chess.Move.from_uci(itos[_ids[_t + 1]]))

# 32 pieces on the board at ply 0, and White to move.
assert _bf[0][OFF_PLANES:OFF_STM].sum() == 32
assert _bf[0][OFF_STM] == 1.0
print("Board feature alignment OK.")

# --- Derived slices must describe the start position correctly ---
_start = board_feature_vector(chess.Board())

if USE_ATTACK_MAPS:
    # At ply 0 the two sides' attack maps are mirror images of each other.
    _wa = _start[OFF_ATTACK:OFF_ATTACK + 64]
    _ba = _start[OFF_ATTACK + 64:OFF_ATTACK + 128]
    assert _wa.sum() == _ba.sum(), "attack maps not symmetric at the start position"
    assert _wa[chess.A3] == 1.0 and _wa[chess.H3] == 1.0, "pawn attacks missing"
    assert _wa[chess.E5] == 0.0, "white should not attack e5 at ply 0"
    print(f"Attack maps OK (white attacks {int(_wa.sum())} squares at ply 0).")

if USE_LEGAL_FROM_TO:
    _lf = _start[OFF_LEGAL:OFF_LEGAL + 64]
    _lt = _start[OFF_LEGAL + 64:OFF_LEGAL + 128]
    # 20 legal opening moves, but only 10 distinct from-squares (8 pawns + 2 knights)
    # and 16 distinct to-squares (ranks 3 and 4; the knight targets already sit on rank 3).
    assert _lf.sum() == 10, f"expected 10 from-squares at ply 0, got {_lf.sum()}"
    assert _lt.sum() == 16, f"expected 16 to-squares at ply 0, got {_lt.sum()}"
    assert _lf[chess.E1] == 0.0, "king has no legal move at ply 0"
    print("Legal from/to OK (10 from-squares, 16 to-squares at ply 0).")

if USE_LAST_MOVE:
    assert _start[OFF_LASTMOVE:OFF_LASTMOVE + 128].sum() == 0, "start position has no last move"
    _b1 = chess.Board(); _b1.push(chess.Move.from_uci("e2e4"))
    _f1 = board_feature_vector(_b1)
    assert _f1[OFF_LASTMOVE + chess.E2] == 1.0 and _f1[OFF_LASTMOVE + 64 + chess.E4] == 1.0
    print("Last-move squares OK.")

if USE_TACTICAL_FLAGS:
    assert _start[OFF_FLAGS] == 0.0, "start position is not check"
    _bc = chess.Board("rnb1kbnr/pppp1ppp/8/4p3/6Pq/5P2/PPPPP2P/RNBQKBNR w KQkq - 1 3")
    assert board_feature_vector(_bc)[OFF_FLAGS] == 1.0, "check flag not set"
    print("Tactical flags OK.")

_m = build_meta_features(_r, _T)
print("\nMeta row 0 [elo_stm, elo_opp, clk_stm, clk_opp, has_clk, tc_base, tc_inc]:")
print(" ", np.round(_m[0], 4))

---
# 6. Dataset & DataLoader

Board planes are built **on the fly** in `__getitem__` rather than precomputed. Precomputing is
tempting but does not scale: 781 float32s x ~1M positions is roughly 3 GB, and the whole point of
this notebook is to stop RAM being the ceiling on data. Replaying a game with `python-chess` costs
about a millisecond, and with `NUM_WORKERS > 0` it overlaps with the GPU.

Padding is masked in **two** places, as before — out of attention via `pad_mask`, and out of the
loss via `-100` targets. Board and meta rows for padded positions are zeros and are attended-out
anyway.

In [ ]:
class ChessMoveDataset(Dataset):
    def __init__(self, records, block_size=BLOCK_SIZE):
        self.block_size = block_size
        self.records = [r for r in records if len(r["ids"]) >= 2]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        ids = rec["ids"][: self.block_size + 1]

        x = ids[:-1]
        y = ids[1:]
        T = len(x)

        board_feats = build_board_features(ids, T)
        meta_feats  = build_meta_features(rec, T)

        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(y, dtype=torch.long),
            torch.from_numpy(board_feats),
            torch.from_numpy(meta_feats),
        )


def collate_batch(batch):
    xs, ys, bfs, mfs = zip(*batch)
    B = len(xs)
    max_len = max(len(x) for x in xs)

    x_pad  = torch.full((B, max_len), PAD_ID, dtype=torch.long)
    y_pad  = torch.full((B, max_len), -100, dtype=torch.long)   # -100 -> ignored by cross-entropy
    bf_pad = torch.zeros((B, max_len, BOARD_DIM), dtype=torch.float32)
    mf_pad = torch.zeros((B, max_len, META_DIM), dtype=torch.float32)

    for i, (x, y, bf, mf) in enumerate(zip(xs, ys, bfs, mfs)):
        n = len(x)
        x_pad[i, :n]  = x
        y_pad[i, :n]  = y
        bf_pad[i, :n] = bf
        mf_pad[i, :n] = mf

    return x_pad, y_pad, bf_pad, mf_pad


train_dataset = ChessMoveDataset(train_records)
test_dataset  = ChessMoveDataset(test_records)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_batch, num_workers=NUM_WORKERS)
val_loader   = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_batch, num_workers=NUM_WORKERS)

print(f"Train samples (games): {len(train_dataset)}")
print(f"Val samples (games):   {len(test_dataset)}")

---
# 7. Model — Board/Meta-Conditioned Decoder-Only Transformer

The transformer stack is byte-for-byte v2's. The only architectural change is what goes *into* the
residual stream at position `t`:

```
x = tok_emb(move_t) + pos_emb(t) + board_proj(board_feats_t) + meta_proj(meta_feats_t)
```

## Input scale balance — the thing that silently breaks this

Only *ratios* between the four input terms matter (the first LayerNorm inside block 1 normalizes
absolute scale away). If one term arrives far larger than the others, the smaller ones are
effectively deleted at initialization.

**Matching the weight init std is not sufficient, and this is easy to get wrong.** Initializing
`board_proj.weight` at `std=0.02` to match the embeddings looks right and is not: roughly **132 board
features are active at once and they sum**, so the output std picks up a factor of `sqrt(132) ~ 11.5`.
Measured on a real game, that put `board_proj` at **12.4x** the token embedding and gave the board
term **98.5% of total input variance** — the move history was all but erased before training
started, and the model had to spend its budget climbing back out.

The fix is scale-invariant rather than hand-tuned: LayerNorm each side-channel to unit scale, then
multiply by a **learnable scalar initialized to `EMB_INIT_STD`**. Every term therefore starts at the
same magnitude regardless of how many features are active, and the model *learns* how much board and
metadata signal to admit. `board_scale` / `meta_scale` are printed after training — if either grows
well above 0.02, the model wanted that channel louder than its starting point.

This also means adding or removing features via the `USE_*` flags no longer silently rebalances the
inputs, which the previous fixed init did.

(v2's own lesson still applies underneath: weight tying plus PyTorch's default `N(0, 1)` embedding
init produces huge logits and destabilizes training from step 1, hence `std=0.02` on the
embeddings.)

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.d_head = d_model // n_head

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(block_size, block_size)).bool()
        self.register_buffer("causal_mask", mask)

    def forward(self, x, pad_mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)

        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~self.causal_mask[:T, :T], float("-inf"))

        if pad_mask is not None:
            att = att.masked_fill(~pad_mask[:, None, None, :], float("-inf"))

        att = self.dropout(F.softmax(att, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class Block(nn.Module):
    def __init__(self, d_model, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, pad_mask=None):
        x = x + self.attn(self.ln1(x), pad_mask)
        x = x + self.mlp(self.ln2(x))
        return x


class ChessGPTv3(nn.Module):
    """Next-move prediction over the full UCI vocabulary, conditioned on the
    actual board state and per-ply player metadata."""

    def __init__(self, vocab_size, board_dim=BOARD_DIM, meta_dim=META_DIM,
                 block_size=BLOCK_SIZE, d_model=256, n_head=4, n_layer=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size

        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_emb = nn.Embedding(block_size, d_model)

        # Each side-channel is LayerNorm'd to unit scale and then multiplied by a
        # LEARNABLE scalar initialized to match the embedding std. See the note below
        # on why a small weight init is not enough on its own.
        self.board_proj  = nn.Linear(board_dim, d_model)
        self.board_ln    = nn.LayerNorm(d_model)
        self.board_scale = nn.Parameter(torch.full((1,), EMB_INIT_STD))

        self.meta_proj  = nn.Linear(meta_dim, d_model)
        self.meta_ln    = nn.LayerNorm(d_model)
        self.meta_scale = nn.Parameter(torch.full((1,), EMB_INIT_STD))

        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [Block(d_model, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=EMB_INIT_STD)
        nn.init.normal_(self.pos_emb.weight, mean=0.0, std=EMB_INIT_STD)
        nn.init.zeros_(self.board_proj.bias)
        nn.init.zeros_(self.meta_proj.bias)
        self.tok_emb.weight.data[PAD_ID].zero_()

    def forward(self, idx, pad_mask=None, board_feats=None, meta_feats=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)

        x = self.tok_emb(idx) + self.pos_emb(pos)
        if board_feats is not None:
            x = x + self.board_scale * self.board_ln(self.board_proj(board_feats))
        if meta_feats is not None:
            x = x + self.meta_scale * self.meta_ln(self.meta_proj(meta_feats))

        x = self.drop(x)
        for block in self.blocks:
            x = block(x, pad_mask)
        return self.head(self.ln_f(x))

In [ ]:
# --- Input balance check. Run this BEFORE training; it is cheap and it catches the
# --- failure mode where one input channel silently swamps the others.
_probe = ChessGPTv3(
    vocab_size=VOCAB_SIZE, board_dim=BOARD_DIM, meta_dim=META_DIM,
    block_size=BLOCK_SIZE, d_model=D_MODEL, n_head=N_HEAD, n_layer=N_LAYER, dropout=0.0,
)
_xb, _yb, _bfb, _mfb = next(iter(train_loader))

with torch.no_grad():
    _tok = _probe.tok_emb(_xb)
    _pos = _probe.pos_emb(torch.arange(_xb.shape[1]).unsqueeze(0))
    _brd = _probe.board_scale * _probe.board_ln(_probe.board_proj(_bfb))
    _met = _probe.meta_scale * _probe.meta_ln(_probe.meta_proj(_mfb))

_terms = {"tok_emb": _tok, "pos_emb": _pos, "board": _brd, "meta": _met}
_tot_var = (_tok + _pos + _brd + _met).var().item()
_ref = _tok.std().item()

print(f"Active board features per position: {_bfb[0].sum(1).mean():.0f} of {BOARD_DIM}")
print(f"\n{'input term':<12} {'std':>9} {'vs tok_emb':>12} {'% of var':>10}")
for _n, _t in _terms.items():
    print(f"{_n:<12} {_t.std().item():>9.4f} {_t.std().item() / _ref:>11.1f}x "
          f"{100 * _t.var().item() / _tot_var:>9.1f}%")

_worst = max(_terms.values(), key=lambda t: t.std().item()).std().item() / _ref
if _worst > 3:
    print(f"\n!! One channel is {_worst:.1f}x the token embedding -- the others are being")
    print("   drowned out at init. Check board_scale / meta_scale.")
else:
    print("\nBalanced: no input channel dominates the residual stream at initialization.")

del _probe, _tok, _pos, _brd, _met

---
# 8. Training

Same recipe as v2 — next-token cross-entropy over the full vocabulary, no legal-move masking during
training, LR warmup then cosine decay stepped per batch, best-val-loss checkpointing — plus **early
stopping**, since v2 established that this dataset size starts overfitting well before epoch 30 and
there is no reason to sit through the rest.

Watch the `val/train ppl ratio` column. Near 1.0 means you are underfitting and should add capacity
or epochs; steadily opening means you are out of data, which at `MAX_GAMES = 15000` is the expected
outcome.

In [ ]:
model = ChessGPTv3(
    vocab_size=VOCAB_SIZE, board_dim=BOARD_DIM, meta_dim=META_DIM,
    block_size=BLOCK_SIZE, d_model=D_MODEL, n_head=N_HEAD,
    n_layer=N_LAYER, dropout=DROPOUT,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

n_params = sum(p.numel() for p in model.parameters())
n_tokens = sum(len(r["ids"]) - 1 for r in train_records)
print(f"Model parameters: {n_params / 1e6:.2f}M")
print(f"Training tokens:  {n_tokens / 1e6:.2f}M  ({n_tokens / n_params:.2f} tokens/param)")
if n_tokens / n_params < 5:
    print("  -> Heavily under-data (compute-optimal is ~20 tokens/param). Expect overfitting;")
    print("     raise MAX_GAMES / add files to PGN_PATHS for the biggest available win.")

total_steps  = N_EPOCHS * len(train_loader)
WARMUP_STEPS = min(200, max(1, total_steps // 20))


def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def run_epoch(loader, train=True, step_scheduler=False):
    model.train(train)
    total_loss, total_tokens = 0.0, 0

    for x, y, bf, mf in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)
        bf, mf = bf.to(device), mf.to(device)
        pad_mask = x != PAD_ID

        with torch.set_grad_enabled(train):
            logits = model(x, pad_mask, board_feats=bf, meta_feats=mf)
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1), ignore_index=-100)

        if train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            if step_scheduler:
                scheduler.step()

        n_tok = (y != -100).sum().item()
        total_loss += loss.item() * n_tok
        total_tokens += n_tok

    return total_loss / max(total_tokens, 1)


history = []
best_val_loss = float("inf")
best_state_dict = None
epochs_since_best = 0

for epoch in range(1, N_EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True, step_scheduler=True)
    val_loss   = run_epoch(val_loader, train=False)
    history.append((epoch, train_loss, val_loss))

    print(
        f"Epoch {epoch}/{N_EPOCHS} | "
        f"train ppl {math.exp(train_loss):.1f} | val ppl {math.exp(val_loss):.1f} | "
        f"val/train ppl ratio {math.exp(val_loss - train_loss):.2f} | "
        f"lr {scheduler.get_last_lr()[0]:.2e}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_dict = {k: v.detach().clone() for k, v in model.state_dict().items()}
        epochs_since_best = 0
    else:
        epochs_since_best += 1
        if epochs_since_best >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stop: no val improvement for {EARLY_STOP_PATIENCE} epochs.")
            break

model.load_state_dict(best_state_dict)
print(f"\nRestored best checkpoint (val loss {best_val_loss:.4f}, ppl {math.exp(best_val_loss):.1f})")

# How loud did the model choose to make each side-channel? Both start at EMB_INIT_STD.
print(f"\nLearned channel gains (init {EMB_INIT_STD}):")
print(f"  board_scale: {model.board_scale.item():.4f}  ({model.board_scale.item() / EMB_INIT_STD:.1f}x init)")
print(f"  meta_scale:  {model.meta_scale.item():.4f}  ({model.meta_scale.item() / EMB_INIT_STD:.1f}x init)")
print("  A gain that grew well above init means the model wanted that channel;")
print("  one that shrank toward 0 means it found the channel unhelpful.")

In [ ]:
epochs, tr_losses, va_losses = zip(*history)

plt.figure(figsize=(7, 4))
plt.plot(epochs, tr_losses, label="Train loss")
plt.plot(epochs, va_losses, label="Val loss")
plt.axvline(int(np.argmin(va_losses)) + 1, color="green", linestyle=":", label="Best checkpoint")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss")
plt.title("Training curve - v3")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

---
# 9. Batched Legal-Move-Masked Evaluation

Same protocol as v1/v2 so the numbers stay comparable, but restructured for speed.

**The change:** v1/v2 called the model once per position, rebuilding the prefix each time — O(n²)
work per game. But evaluation is teacher-forced on the *real* move history, and causal masking
guarantees that the logits at position `t` are already the prediction for `t+1` given only
`≤ t`. So **one forward pass per game** yields every position at once, with no leakage. This is
roughly a 40x speedup, which is what makes `EVAL_MAX_GAMES = 1500` (~120k positions) affordable
where v1 evaluated 3,000.

That sample size matters: at 3,000 positions the standard error on Top-1 is about ±0.8%, so the
earlier setup could not reliably resolve the 1–2% differences between model variants.

Legality still comes from `python-chess` replaying the **actual** game, never from the model's
belief about the board.

**A note on `unmasked_argmax_legal_rate`.** In v1/v2 this was a board-*tracking* diagnostic: 0.539
meant the model's own top pick was illegal 46% of the time. In v3 the board is an input, so this
number should jump sharply — it no longer measures tracking, it measures how well the model learned
to read legality off the planes it was handed. Expect it to rise a lot, and do not read that rise
as the accuracy improvement.

In [ ]:
@torch.no_grad()
def evaluate_masked_topk(records, k_values=(1, 3), max_games_eval=None, desc="Evaluating"):
    """One forward pass per game (teacher-forced), then mask to legal moves per position."""
    model.eval()

    hits = {k: 0 for k in k_values}
    total = 0
    unmasked_legal_argmax = 0

    subset = records if max_games_eval is None else records[:max_games_eval]

    for rec in tqdm(subset, desc=desc):
        ids = rec["ids"][: BLOCK_SIZE + 1]
        if len(ids) < 2:
            continue
        T = len(ids) - 1

        # Single replay yields both the board features and the per-position legal moves.
        board_feats = np.zeros((T, BOARD_DIM), dtype=np.float32)
        legal_per_pos = []
        board = chess.Board()
        for t in range(T):
            board_feature_vector(board, out=board_feats[t])
            legal_per_pos.append([stoi[m.uci()] for m in board.legal_moves])
            board.push(chess.Move.from_uci(itos[ids[t + 1]]))

        meta_feats = build_meta_features(rec, T)

        x  = torch.tensor(ids[:-1], dtype=torch.long, device=device).unsqueeze(0)
        bf = torch.from_numpy(board_feats).unsqueeze(0).to(device)
        mf = torch.from_numpy(meta_feats).unsqueeze(0).to(device)
        pad_mask = torch.ones_like(x, dtype=torch.bool)

        logits = model(x, pad_mask, board_feats=bf, meta_feats=mf)[0]      # (T, V)
        probs = F.softmax(logits.float(), dim=-1).cpu().numpy()

        for t in range(T):
            p = probs[t]
            legal_ids = legal_per_pos[t]
            if not legal_ids:
                continue

            # Diagnostic: was the model's raw top pick legal, before masking?
            if int(p.argmax()) in set(legal_ids):
                unmasked_legal_argmax += 1

            legal_probs = p[legal_ids]
            ranked = [legal_ids[i] for i in np.argsort(-legal_probs)]
            played_id = ids[t + 1]

            for k in k_values:
                if played_id in ranked[:k]:
                    hits[k] += 1
            total += 1

    results = {f"top_{k}": hits[k] / max(1, total) for k in k_values}
    results["unmasked_argmax_legal_rate"] = unmasked_legal_argmax / max(1, total)
    results["positions_evaluated"] = total
    return results


def report(name, res):
    print(f"{name}:")
    for k, v in res.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


t0 = time.time()
eval_test = evaluate_masked_topk(test_records, max_games_eval=EVAL_MAX_GAMES, desc="Test")
report("Held-out (test) games", eval_test)

eval_train = evaluate_masked_topk(train_records, max_games_eval=EVAL_MAX_GAMES, desc="Train")
report("\nTrain games (same sample size, for the overfit gap)", eval_train)

print(f"\nTrain/test Top-1 gap: {eval_train['top_1'] - eval_test['top_1']:.4f}")
print(f"Evaluation took {time.time() - t0:.1f}s for "
      f"{eval_test['positions_evaluated'] + eval_train['positions_evaluated']} positions")

In [ ]:
# --- Comparison table. Fill in your own v2 numbers if they differ. ---
V1_TEST = {"top_1": 0.2693, "top_3": 0.4393, "unmasked_argmax_legal_rate": 0.5393}
V2_TEST = {"top_1": 0.29,   "top_3": None,   "unmasked_argmax_legal_rate": None}

rows = [("v1 (5 epochs)", V1_TEST), ("v2 (ELO avg)", V2_TEST), ("v3 (board+meta)", eval_test)]

print(f"{'Model':<20} {'Top-1':>8} {'Top-3':>8} {'unmasked legal':>16}")
print("-" * 56)
for name, r in rows:
    def fmt(v):
        return f"{v:.4f}" if isinstance(v, float) else "     -"
    print(f"{name:<20} {fmt(r.get('top_1')):>8} {fmt(r.get('top_3')):>8} "
          f"{fmt(r.get('unmasked_argmax_legal_rate')):>16}")

---
# 10. Save Artifacts

Written to **v3-specific filenames**. v1 and v2 both save to `chess_gpt_baseline.pt` /
`chess_move_vocab.json`, so running either notebook's save cell overwrites the other's weights in
place — v3 stays out of that collision.

The config is saved alongside the weights because the checkpoint is not loadable without it: the
tied head, `BOARD_DIM`, and `META_DIM` all have to match exactly at load time.

In [ ]:
torch.save(model.state_dict(), CKPT_PATH)

with open(VOCAB_PATH, "w") as f:
    json.dump({
        "itos": itos,
        "block_size": BLOCK_SIZE,
        "board_dim": BOARD_DIM,
        "meta_dim": META_DIM,
        "d_model": D_MODEL,
        "n_head": N_HEAD,
        "n_layer": N_LAYER,
        "min_elo": MIN_ELO,
        "max_elo": MAX_ELO,
        "has_clock_data": bool(HAS_CLOCK_DATA),
    }, f)

print(f"Saved weights to     {CKPT_PATH}")
print(f"Saved vocab/config to {VOCAB_PATH}")

---
# 11. Summary & Next Steps

## What v3 adds

| Component | Detail |
|---|---|
| **Input** | Move tokens + board state & derived features (1167 dims) + per-ply metadata (7 dims) |
| **Derived features** | Attack maps, legal from/to squares, last-move squares, check + halfmove clock — each individually toggleable for ablation |
| **Metadata** | Side-to-move ELO, opponent ELO, time control base/increment, causally-lagged clocks (inert on Elite data) |
| **Model** | v2's stack + `board_proj` / `meta_proj`, all residual inputs init'd at `std=0.02` |
| **Training** | Warmup + cosine, best-val checkpointing, early stopping, dropout 0.2 |
| **Evaluation** | One forward pass per game, ~40x faster, ~120k positions instead of 3k |

## Read the results in this order

1. **Test Top-1 vs v2's 0.29.** The headline. Board state should be the bulk of any gain.
2. **Train/test Top-1 gap.** If it is still wide, v3 did not fix the overfit — and it was never
   going to on its own. Data is the fix.
3. **`unmasked_argmax_legal_rate`.** Expect a large jump for a structural reason (the board is now
   an input), not as evidence of better move choice. Do not report it as the improvement.

## Next levers, in order

1. **More data.** Still the largest available win by a wide margin, and the only real fix for the
   overfit. Add months to `PGN_PATHS` and raise `MAX_GAMES` toward ~1M games. Everything below is
   worth less than this.
2. **Color-relative encoding.** Flip the board on Black's turns (`sq ^ 56`) and remap the move
   tokens so the model learns one skill instead of two mirror-image copies. Watch castling and
   promotion tokens through the remap.
3. **Factorized policy head.** Predict `from_square` and `to_square` separately instead of one flat
   softmax over 4,546 tokens; shares statistical strength across moves touching the same squares.
   Legal masking still works — score each legal `(from, to)` pair as the sum of the two logits.
4. **Ablate the parts of v3.** The `USE_*` flags in Section 1 exist for this. The one I would test
   first is `USE_LEGAL_FROM_TO`: it is by far the most expensive feature to build (~40 us/position,
   more than everything else combined) so it should have to justify itself.
5. **Stockfish eval as a feature.** Genuinely new information rather than a re-encoding of the
   position, but you pay engine time on every training *and* inference position. Only after 1–3.
6. **Bigger model.** Last. At the current tokens/param ratio, more capacity overfits sooner.

## Never

Do not condition on `Result`, `WhiteRatingDiff` / `BlackRatingDiff`, `Termination`, `ECO` /
`Opening`, or any post-move engine evaluation. All of them are future information relative to the
move being predicted, and all of them will inflate these metrics into meaninglessness. The full
reasoning is in the Section 5 leakage table.